# Run DSA4213 Assignment 1 on Colab

This notebook is just a **runner**: it does not contain any model code itself. It sets up a Colab GPU machine, pulls the actual code from the GitHub repo, installs the dependencies, and gives you one empty cell per training command you will run later.

**Before you start:** in the Colab menu, go to `Runtime > Change runtime type` and set `Hardware accelerator` to `GPU` (T4 is fine). The next cell checks that this worked.

In [ ]:
# Check that Colab actually gave us a GPU.
# `nvidia-smi` is a command-line tool that reports on NVIDIA GPUs; if no GPU is
# attached, this command itself will fail with "command not found".
!nvidia-smi

# This double-checks from inside PyTorch, which is what actually matters for
# training: torch.cuda.is_available() returns True only if PyTorch can see and
# use a GPU. If this prints False, go back and change the runtime type above,
# then re-run this cell (Runtime > Restart runtime if needed).
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

## Clone the repository

This downloads a fresh copy of the code from GitHub onto the Colab machine (Colab's disk is temporary — it is wiped when the session ends, so you clone from scratch every time).

`REPO_URL` below is already set to your repo's HTTPS address. Make sure your latest commits are pushed to GitHub first.

In [ ]:
REPO_URL = "https://github.com/izzryanshah/dsa4213-assignment1-izzryanshah.git"  # HTTPS URL (Colab cannot use the SSH one). If the repo is private, use a personal access token: https://<token>@github.com/izzryanshah/dsa4213-assignment1-izzryanshah.git
REPO_DIR = "dsa4213-assignment1"

# -d checks whether the folder already exists, so re-running this cell in the
# same session doesn't error out trying to clone into an existing directory.
import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists, skipping clone.")

# %cd (a Colab/Jupyter "magic" command) changes the notebook's working
# directory, persisting for every cell after this one — unlike !cd, which
# would only affect that one shell command.
%cd {REPO_DIR}

## Install dependencies

Installs everything listed in `requirements.txt` (torch, numpy, nltk, gensim, scikit-learn, matplotlib, pandas) into this Colab session. Colab already ships with most of these, so this mainly fills in anything missing or on the wrong version.

In [ ]:
!pip install -r requirements.txt

## Save results to Google Drive (so Colab runs appear in RESULTS.md)

Colab's disk is wiped when the session ends, and every session starts from a fresh clone. The cell below connects Google Drive and defines a function `save_results()`. You call it (via the cell that follows each training cell) to:

1. run `src/update_results.py`, which rebuilds `RESULTS.md` and `results/figures/` from everything in `results/`;
2. copy `results/` and `RESULTS.md` to your Drive folder.

At the start of a session it also copies the previous session's `results/` and `RESULTS.md` back from Drive, so the rebuilt `RESULTS.md` contains *all* runs, and the text you wrote under the *My notes* headings is kept. (`rsync -au` only overwrites a file when the Drive copy is newer.)

In [ ]:
# Mounts your Google Drive at /content/drive; files written there are saved to your real Drive.
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DEST = "/content/drive/MyDrive/Education/University/dsa4213-assignment1"  # your project folder on Drive; results/, checkpoints/ and RESULTS.md are saved here
!mkdir -p {DRIVE_DEST}/results

# Restore earlier sessions' results and RESULTS.md (if any) into this fresh clone.
# -a keeps file details, -u skips files that are newer in the clone than on Drive.
!rsync -au {DRIVE_DEST}/results/ results/
!mkdir -p checkpoints && rsync -au {DRIVE_DEST}/checkpoints/ checkpoints/
!test -f {DRIVE_DEST}/RESULTS.md && rsync -u {DRIVE_DEST}/RESULTS.md RESULTS.md || echo "No RESULTS.md on Drive yet."

def save_results():
    """Rebuild RESULTS.md from results/*.json, then copy results/ and RESULTS.md to Drive."""
    !python src/update_results.py --config configs/update_results.json
    !python src/summarise_runs.py
    !rsync -a results/ {DRIVE_DEST}/results/
    !mkdir -p checkpoints {DRIVE_DEST}/checkpoints
    !rsync -a checkpoints/ {DRIVE_DEST}/checkpoints/
    !cp RESULTS.md {DRIVE_DEST}/RESULTS.md
    print(f"Saved results/ and RESULTS.md to {DRIVE_DEST}")

## Training commands

Run the cells in order. Each training cell is followed by a `save_results()` cell that rebuilds `RESULTS.md` and copies `results/` and `checkpoints/` to Drive. The `-u` flag in the commands makes the epoch lines print live.

**Colab has no resume:** if the session disconnects mid-run, that run is lost. Train one model per session if you can, and keep the tab open.

### 1. Copy the processed data from Drive

`data/` is not in GitHub, so the clone has no data. Before the first session, copy your laptop's `data/processed/` folder (55 MB) to Google Drive at the path below, so the train/validation/test splits are exactly the ones you made once and saved. Change `DATA_DRIVE_DIR` if you used another folder.

In [ ]:
DATA_DRIVE_DIR = "/content/drive/MyDrive/Education/University/dsa4213-assignment1/data/processed"  # <-- put your laptop's data/processed/ folder here

!mkdir -p data
!rsync -a {DATA_DRIVE_DIR}/ data/processed/
!ls -la data/processed   # expect: train.npz, val.npz, test.npz, vocab.json, label_map.json, split_assignment.csv

### 2. Train the LSTM

Settings come from `configs/train_lstm.json` (hidden size, layers, dropout, learning rate, ...). To change one for this run, add a flag, e.g. `--hidden-size 512`; the run record saves the values actually used.

In [ ]:
!python -u src/train.py --config configs/train_lstm.json --model lstm

*Run the next cell after the training cell above: it rebuilds `RESULTS.md` and saves `results/` to Drive.*

In [ ]:
save_results()

### 3. Train the RNN

Settings come from `configs/train_rnn.json`.

In [ ]:
!python -u src/train.py --config configs/train_rnn.json --model rnn

*Run the next cell after the training cell above: it rebuilds `RESULTS.md` and saves `results/` to Drive.*

In [ ]:
save_results()

### 4. Train the Transformer

Settings come from `configs/train_transformer.json` (2 blocks, 4 heads, feed-forward width 256, `max_positions` 256, dropout 0.3). To change one for this run, add a flag, e.g. `--layers 4`; the run record saves the values actually used.

In [ ]:
!python -u src/train.py --config configs/train_transformer.json --model transformer

*Run the next cell after the training cell above: it rebuilds `RESULTS.md` and saves `results/` and `checkpoints/` to Drive.*

In [ ]:
save_results()

### 5. Benchmark inference speed (after the RNN, LSTM and Transformer have trained)

Times every model on the whole test set (batch size 64, context length 256, 2 warm-up passes, 3 timed repeats) on the Colab GPU and CPU, and writes `results/part1_costs.csv` plus `results/benchmark.json`. It needs `checkpoints/train_rnn.pt`, `train_lstm.pt` and `train_transformer.pt` (copy them back from Drive if this is a fresh session) and the run records in `results/`. Run it on the same Colab machine type every time.

In [ ]:
!python -u src/benchmark.py --config configs/benchmark.json

*Run the next cell after the benchmark cell above: it rebuilds `RESULTS.md` and saves `results/` to Drive.*

In [ ]:
save_results()

### 6. Generate text from the RNN, LSTM and Transformer

Same prompts (`prompts.txt`) and identical decoding settings in all three configs (temperature 1.0, top-k 40, 50 new tokens, seed 0, 1 sample; the n-gram file in the repo already uses them). Needs `checkpoints/train_rnn.pt`, `train_lstm.pt` and `train_transformer.pt`. Each run writes `results/generations_<model>.json`.

In [ ]:
!python -u src/generate.py --config configs/generations_rnn.json
!python -u src/generate.py --config configs/generations_lstm.json
!python -u src/generate.py --config configs/generations_transformer.json

Now merge all four models into one side-by-side file, `results/part1_generations.md`. It stops with an error if any model's decoding settings or prompts differ.

In [ ]:
!python -u src/combine_generations.py --config configs/combine_generations.json

*Run the next cell after the cells above: it rebuilds `RESULTS.md` and saves `results/` to Drive.*

In [ ]:
save_results()

### 7. Re-run the bigram baseline on the Colab GPU

The bigram baseline was first run on the laptop CPU, but neural training must come from Colab. This re-run uses a new run name (`train_bigram_colab`), so the local `results/train_bigram.json` is kept. Settings come from `configs/train_bigram.json`.

In [ ]:
!python -u src/train.py --config configs/train_bigram.json --model bigram --run-name train_bigram_colab

*Run the next cell after the training cell above: it rebuilds `RESULTS.md` and saves `results/` and `checkpoints/` to Drive.*

In [ ]:
save_results()

`train.py` already evaluates the best checkpoint on the test set once, at the end of each run, so there is no separate evaluation step. Your results are in `results/<run>.json`, copied to Drive by `save_results()`. Afterwards, copy `results/` and `RESULTS.md` from Drive back into your laptop's repo.

## Pulling new commits

If you change code on your own laptop and push it to GitHub, this Colab session's clone does **not** update automatically — you have to pull manually. Run `!git pull` from inside the repo folder (the `%cd {REPO_DIR}` cell above already put you there). If Colab reports local changes that conflict with the pull (e.g. because `results/` files were modified here), either commit them first or discard them with `!git checkout -- .`, then pull again.

Saving `results/` to Drive is handled by the `save_results()` cell near the top of this notebook.